# HyCF-TFM: 基于 Transformer 融合的纵向多模态心衰风险预测框架

本 Notebook 实现了 HyCF-TFM 模型的实验环境与核心代码结构。该框架旨在融合文本、时序数据和影像数据，用于预测心衰患者出院后 30 天内的再入院风险。

## 1. 实验环境准备
首先安装必要的依赖库。

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import wfdb
import cv2
from PIL import Image
from torchvision import transforms, models
from openai import OpenAI
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score
import matplotlib.pyplot as plt
import glob

# Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Setup OpenAI API
API_KEY = "REDACTED_CONFIGURE_LOCALLY"
BASE_URL = "https://yunwu.ai/v1"

client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
)

print("Libraries imported and Environment Configured.")

In [ ]:
class MultimodalDataset(Dataset):
    def __init__(self, cpsc_root, physionet_root, img_folder, mimic_root, extra_ecg_roots, supplementary_csv, transform=None):
        """
        cpsc_root: CPSC2021 ECG path
        physionet_root: PhysioNet ECG path
        img_folder: X-Ray images path
        mimic_root: MIMIC-III CSV folder
        extra_ecg_roots: Dict or path for extra ECGs
        supplementary_csv: Path to single external CSV (e.g., Heart_failure_clinical_records_dataset.csv)
        """
        self.transform = transform
        self.samples = []
        
        # --- Statistics Containers ---
        stats = {
            "image_count": 0,
            "ecg_sources": {},
            "total_ecg": 0,
            "text_sources": {"MIMIC-III": 0, "SupplementaryCSV": 0},
            "total_text": 0
        }

        # ==========================================
        # 1. Text & Label Source A: MIMIC-III (Comprehensive)
        # ==========================================
        print(f"Scanning MIMIC-III Data from {mimic_root}...")
        self.text_samples = self._load_mimic_comprehensive(mimic_root)
        stats["text_sources"]["MIMIC-III"] = len(self.text_samples)

        # ==========================================
        # 1. Text & Label Source B: Supplementary CSV
        # ==========================================
        if os.path.exists(supplementary_csv):
            print(f"Scanning Supplementary CSV from {supplementary_csv}...")
            try:
                df_supp = pd.read_csv(supplementary_csv)
                # Convert structured data to natural language text for the Text Engine
                # Columns: age, anaemia, diabetes, ..., DEATH_EVENT
                
                count = 0
                for _, row in df_supp.iterrows():
                    # Construct valid clinical sentence
                    # text = "Patient is 75 years old. Hx of Anaemia: No. Diabetes: No. EF: 20%..."
                    desc_parts = []
                    desc_parts.append(f"Patient age {row.get('age', 'N/A')}.")
                    if row.get('anaemia', 0) == 1: desc_parts.append("History of Anaemia.")
                    if row.get('diabetes', 0) == 1: desc_parts.append("History of Diabetes.")
                    if row.get('high_blood_pressure', 0) == 1: desc_parts.append("Hypertension present.")
                    if row.get('smoking', 0) == 1: desc_parts.append("Smoker.")
                    desc_parts.append(f"Ejection Fraction: {row.get('ejection_fraction', 'N/A')}%.")
                    desc_parts.append(f"Serum Creatinine: {row.get('serum_creatinine', 'N/A')}.")
                    desc_parts.append(f"Serum Sodium: {row.get('serum_sodium', 'N/A')}.")
                    
                    full_text = " ".join(desc_parts)
                    
                    # Target Mapping: DEATH_EVENT (1=Dead) roughly correlates with High Risk outcomes like Readmission
                    # User asked to integrate/align. We treat 'Bad Outcome' as class 1.
                    risk_label = int(row.get('DEATH_EVENT', 0))
                    
                    self.text_samples.append({
                        'text': full_text,
                        'label': risk_label,
                        'diagnosis': "Heart Failure (Structured)",
                        'source': 'SuppCSV'
                    })
                    count += 1
                
                stats["text_sources"]["SupplementaryCSV"] = count
                print(f"  -> Integrated {count} Structured Records as Text.")
                
            except Exception as e:
                print(f"Error parsing Supplementary CSV: {e}")

        # Fallback
        if not self.text_samples:
            self.text_samples = [{'text': "No data.", 'label': 0, 'diagnosis': 'Unknown', 'source': 'Dummy'}]
        
        stats["total_text"] = len(self.text_samples)

        # ==========================================
        # 2. Load ECG Data (All Sources)
        # ==========================================
        self.ecg_files = []
        
        def scan_ecg_folder(folder_path, source_name):
            found_paths = []
            if isinstance(folder_path, str) and os.path.exists(folder_path):
                # Search for .hea recursively
                headers = glob.glob(os.path.join(folder_path, "**", "*.hea"), recursive=True)
                for h in headers:
                    base = os.path.splitext(h)[0]
                    # Check for signal file (.mat, .dat, .ecg)
                    if any(os.path.exists(base + ext) for ext in ['.mat', '.dat', '.ecg']):
                        found_paths.append(base)
                
                cnt = len(found_paths)
                stats["ecg_sources"][source_name] = cnt
                print(f"  -> Found {cnt} ECGs in {source_name}")
            return found_paths

        # A. PhysioNet 2017
        self.ecg_files.extend(scan_ecg_folder(physionet_root, "PhysioNet2017"))
        # B. CPSC 2021
        self.ecg_files.extend(scan_ecg_folder(cpsc_root, "CPSC2021"))
        # C. Extra (Congestive, Files)
        if isinstance(extra_ecg_roots, dict):
            for name, path in extra_ecg_roots.items():
                self.ecg_files.extend(scan_ecg_folder(path, name))
        elif isinstance(extra_ecg_roots, str):
             self.ecg_files.extend(scan_ecg_folder(extra_ecg_roots, "ExtraFiles"))
             
        stats["total_ecg"] = len(self.ecg_files)

        # ==========================================
        # 3. Load Images (Base Anchor)
        # ==========================================
        self.image_files = []
        print(f"Scanning Images from {img_folder}...")
        for ext in ['*.jpg', '*.png', '*.jpeg']:
            self.image_files.extend(glob.glob(os.path.join(img_folder, "**", ext), recursive=True))
        stats["image_count"] = len(self.image_files)
        print(f"  -> Found {stats['image_count']} Images.")
        
        # ==========================================
        # 4. Alignment
        # ==========================================
        # Alignment Strategy: Image Anchor -> Random Text (Label) -> Random ECG
        
        for img_path in self.image_files:
            # 1. Text/Label Sample (from Combined Pool)
            txt_sample = np.random.choice(self.text_samples)
            
            # 2. ECG Path
            ecg_path = np.random.choice(self.ecg_files) if self.ecg_files else None
                
            self.samples.append({
                'image_path': img_path,
                'real_img_path': img_path,
                'ecg_path': ecg_path,
                'text': txt_sample['text'],
                'label': txt_sample['label'],
                'diagnosis_summary': txt_sample.get('diagnosis', 'Unknown'),
                'source': txt_sample['source']
            })
            
        print("="*40)
        print("Final Dataset Statistics:")
        print(f"Image Count:       {stats['image_count']}")
        print(f"Text Pool Size:    {stats['total_text']} (MIMIC: {stats['text_sources']['MIMIC-III']}, Supp: {stats['text_sources']['SupplementaryCSV']})")
        print(f"ECG Pool Size:     {stats['total_ecg']}")
        print(f"Total Constructed: {len(self.samples)}")
        print("="*40)

    def _load_mimic_comprehensive(self, mimic_root):
        """
        Parses MIMIC-III CSVs to build a rich narrative representation including:
        - Demographics (PATIENTS)
        - Admission Info (ADMISSIONS)
        - Diagnosis Names (DIAGNOSES_ICD + D_ICD_DIAGNOSES)
        - Discharge Summaries (NOTEEVENTS)
        - ICU Stays (ICUSTAYS)
        Returns: List of dicts {'text', 'label', 'source'}
        """
        extracted_samples = []
        try:
            # 1. ADMISSIONS (Base)
            df_adm = pd.read_csv(os.path.join(mimic_root, "ADMISSIONS.csv"))
            # Coerce errors to handle potential parse issues
            df_adm['admittime'] = pd.to_datetime(df_adm['admittime'], errors='coerce')
            df_adm['dischtime'] = pd.to_datetime(df_adm['dischtime'], errors='coerce')
            
            # Label Gen
            df_adm = df_adm.sort_values(by=['subject_id', 'admittime'])
            df_adm['next_admittime'] = df_adm.groupby('subject_id')['admittime'].shift(-1)
            
            # Calculate gap safely
            df_adm['gap'] = (df_adm['next_admittime'] - df_adm['dischtime']).dt.total_seconds() / 86400
            df_adm['readmission_30d'] = ((df_adm['gap'] >= 0) & (df_adm['gap'] <= 30)).astype(int)
            
            # 2. PATIENTS (Demographics)
            df_pat = pd.read_csv(os.path.join(mimic_root, "PATIENTS.csv"))
            df_pat['dob'] = pd.to_datetime(df_pat['dob'], errors='coerce')
            df_main = pd.merge(df_adm, df_pat[['subject_id', 'gender', 'dob']], on='subject_id', how='left')
            
            # Calculate Age SAFELY (Avoid OverflowError for >89y patients where DoB is shifted 300 years)
            # Instead of (admittime - dob).dt.days which overflows for ~300 years diff, use Year difference
            df_main['age_raw'] = df_main['admittime'].dt.year - df_main['dob'].dt.year
            
            def process_age(x):
                # MIMIC-III convention: >89 are shifted to 300+ years old. 
                # We clip them to 90 or 91.4 (median)
                if pd.isna(x): return 0
                if x > 120: return 91.4
                return x
                
            df_main['age'] = df_main['age_raw'].apply(process_age)
            
            # 3. NOTEEVENTS (Text)
            df_notes = pd.read_csv(os.path.join(mimic_root, "NOTEEVENTS.csv"))
            # Prioritize Discharge Summaries
            df_disch = df_notes[df_notes['category'] == 'Discharge summary'].groupby('hadm_id')['text'].apply(lambda x: " ".join(x)).reset_index()
            df_main = pd.merge(df_main, df_disch, on='hadm_id', how='left')
            
            # 4. DIAGNOSES (Clinical Context)
            if os.path.exists(os.path.join(mimic_root, "DIAGNOSES_ICD.csv")) and os.path.exists(os.path.join(mimic_root, "D_ICD_DIAGNOSES.csv")):
                df_diag = pd.read_csv(os.path.join(mimic_root, "DIAGNOSES_ICD.csv"))
                df_d_icd = pd.read_csv(os.path.join(mimic_root, "D_ICD_DIAGNOSES.csv"))
                
                # Merge to get long titles
                df_diag_full = pd.merge(df_diag, df_d_icd[['icd9_code', 'long_title']], on='icd9_code', how='left')
                
                # Aggregate diagnoses by admission
                def format_diag(row):
                    title = str(row['long_title'])
                    return title
                    
                df_diag_full['diag_str'] = df_diag_full.apply(format_diag, axis=1)
                
                # Collect all diagnoses per admission
                df_diag_agg = df_diag_full.groupby('hadm_id')['diag_str'].apply(lambda x: "; ".join(x.dropna().astype(str))).reset_index()
                df_diag_agg.rename(columns={'diag_str': 'diagnosis_list'}, inplace=True)
                
                df_main = pd.merge(df_main, df_diag_agg, on='hadm_id', how='left')
            
            # 5. ICUSTAYS (LOS)
            if os.path.exists(os.path.join(mimic_root, "ICUSTAYS.csv")):
                df_icu = pd.read_csv(os.path.join(mimic_root, "ICUSTAYS.csv"))
                df_icu_agg = df_icu.groupby('hadm_id')['los'].sum().reset_index()
                df_main = pd.merge(df_main, df_icu_agg, on='hadm_id', how='left')

            # --- Construct Final Text ---
            for _, row in df_main.iterrows():
                # Filter: Only keep valid entries (e.g. must have some text or diagnosis)
                if pd.isna(row.get('text')) and pd.isna(row.get('diagnosis_list')):
                    continue
                    
                # Build Prompt-like String
                txt_parts = []
                # Part 1: Demo
                age_val = row['age']
                txt_parts.append(f"Patient info: {row['gender']} {age_val:.1f}yo.")
                if pd.notnull(row.get('los')): 
                    txt_parts.append(f"ICU Stay: {row['los']:.1f} days.")
                
                # Part 2: Diagnosis
                diags = row.get('diagnosis_list', '')
                if pd.notnull(diags) and len(str(diags)) > 5:
                    txt_parts.append(f"\nDiagnoses: {str(diags)[:500]}.") # Truncate diagnosis list if too long
                
                # Part 3: Narrative
                note = row.get('text', '')
                if pd.notnull(note) and len(str(note)) > 5:
                    clean_note = str(note).replace('\n', ' ').replace('\r', ' ')
                    txt_parts.append(f"\nClinical Note: {clean_note[:2000]}") # Truncate note
                
                final_text = " ".join(txt_parts)
                
                # Label
                label = int(row['readmission_30d'])
                
                extracted_samples.append({
                    'text': final_text,
                    'label': label,
                    'diagnosis': row.get('diagnosis', 'MIMIC Comprehensive'),
                    'source': 'MIMIC-III'
                })
                
            print(f"  -> MIMIC Comprehensive Parser: Successfully processed {len(extracted_samples)} admissions.")
            
        except Exception as e:
            print(f"Critical Error in MIMIC Comprehensive Parser: {e}")
            import traceback
            traceback.print_exc()
            
        return extracted_samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        
        # --- A. ECG (WFDB) ---
        target_len = 2000
        ecg_data = np.zeros(target_len, dtype=np.float32)
        if sample['ecg_path']:
            try:
                sig, meta = wfdb.rdsamp(sample['ecg_path'], channels=[0])
                if sig is not None and len(sig) > 0:
                    raw = sig.flatten()
                    if np.std(raw) > 1e-4 and not np.isnan(raw).any():
                        raw = (raw - np.mean(raw)) / (np.std(raw) + 1e-6)
                        if len(raw) >= target_len:
                            start = np.random.randint(0, len(raw) - target_len)
                            ecg_data = raw[start : start + target_len]
                        else:
                            ecg_data[:len(raw)] = raw
            except: pass
        ecg_tensor = torch.tensor(ecg_data, dtype=torch.float32).unsqueeze(0)

        # --- B. Image ---
        try:
            image = Image.open(sample['image_path']).convert('RGB')
            if self.transform:
                image = self.transform(image)
        except:
            image = torch.zeros(3, 224, 224)

        # --- C. Text ---
        text_raw = str(sample['text'])[:2000]
        
        return {
            "ecg": ecg_tensor,
            "image": image,
            "text_raw": text_raw,
            "label": torch.tensor(float(sample['label']), dtype=torch.float32),
            "nyha": str(sample['diagnosis_summary']),
            "img_risk": f"Source: {sample['source']}",
            "text": text_raw[:200]
        }

In [ ]:
# Instantiation Paths
physionet_root = r"e:\平时作业\人工智能\data\af-classification-from-a-short-single-lead-ecg-recording-the-physionet-computing-in-cardiology-challenge-2017-1.0.0"
cpsc_root = r"e:\平时作业\人工智能\data\paroxysmal-atrial-fibrillation-events-detection-from-dynamic-ecg-recordings-the-4th-china-physiological-signal-challenge-2021-1.0.0"
img_path_root = r"e:\平时作业\人工智能\data\images\chexchonet-a-chest-radiograph-dataset-with-gold-standard-echocardiography-labels-1.0.0"
mimic_root = r"e:\平时作业\人工智能\data\mimic-iii-clinical-database-demo-1.4"
supp_csv_path = r"e:\平时作业\人工智能\data\Heart_failure_clinical_records_dataset.csv"

# Extra ECG Paths Map
extra_ecg = {
    "CongestiveHF": r"e:\平时作业\人工智能\data\congestive-heart-failure-rr-interval-database-1.0.0",
    "FilesDB": r"e:\平时作业\人工智能\data\files"
}

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print("Initializing Fully Integrated Multimodal Dataset...")
dataset = MultimodalDataset(
    cpsc_root=cpsc_root, 
    physionet_root=physionet_root, 
    img_folder=img_path_root, 
    mimic_root=mimic_root,
    extra_ecg_roots=extra_ecg,
    supplementary_csv=supp_csv_path, # New Param layer
    transform=transform
)

# Split
print("\nPreparing Train/Test Splits...")
if len(dataset) > 0:
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)
    print(f"Data Loaded. Train: {len(train_dataset)}, Test: {len(test_dataset)}")
else:
    print("Error: Dataset empty.")

In [ ]:
# --- Data Verification & Visualization ---
import seaborn as sns

def visualize_data_samples(dataset):
    print("--- Visualizing Dataset Samples ---")
    
    if len(dataset) == 0:
        print("Dataset is empty.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Randomly pick samples
    indices = np.random.choice(len(dataset), min(4, len(dataset)), replace=False)
    
    for i, idx in enumerate(indices):
        sample = dataset[idx]
        ax = axes[i//2, i%2]
        
        # Plot Image
        img_np = sample['image'].permute(1, 2, 0).numpy()
        # Un-normalize for display
        img_np = img_np * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img_np = np.clip(img_np, 0, 1)
        
        ax.imshow(img_np)
        
        # Overlay ECG Signal snippet
        ecg = sample['ecg'].numpy().flatten()
        # Normalize ecg to fit in image window (visual trick)
        if ecg.max() != ecg.min():
            ecg_vis = (ecg[:224] - ecg.min()) / ((ecg.max() - ecg.min()) + 1e-6) * 50 + 150 
            ax.plot(ecg_vis, color='red', linewidth=1, label="ECG Lead I")
        
        # Info
        label_txt = "Readmit" if sample['label'].item() == 1.0 else "No Readmit"
        diag = str(sample['nyha'])[:30] # Diagnosis summary
        
        # Wrap text for display
        txt_preview = str(sample['text_raw'])[:50].replace('\n', ' ') + "..."
        
        ax.set_title(f"Sample {idx}: {label_txt}\nDx: {diag}\nNote: {txt_preview}", fontsize=9)
        ax.axis('off')
        ax.legend(loc="lower right")

    plt.tight_layout()
    plt.show()

    # Distribution Plot
    labels = [dataset[i]['label'].item() for i in range(len(dataset))]
    plt.figure(figsize=(6, 4))
    sns.countplot(x=labels, hue=labels, palette="viridis", legend=False)
    plt.title("Readmission Risk Distribution (MIMIC-III Label)")
    plt.xlabel("Label (0=No, 1=Yes)")
    plt.ylabel("Count")
    plt.show()

# Run Visualization
if len(dataset) > 0:
    visualize_data_samples(dataset)

## 2. 数据集定义 (Mock Dataset)

在实际实验中，您需要从 MIMIC-IV 或 eICU 提取数据。这里我们构建一个 `MockHyCFDataset` 来模拟输入数据格式。

**数据格式说明：**
- **Text (出院小结)**: 字符串形式的临床文本。
- **Time-Series (实验室指标)**: 形状为 `(Seq_Len, Num_Features)` 的张量。例如：出院前 7 天，每天 10 个指标 (BNP, 肌酐等)。
- **Image (胸部 X 光)**: 形状为 `(3, 224, 224)` 的图像张量。
- **Label**: 0 (未再入院) 或 1 (再入院)。

In [ ]:
from transformers import AutoModel, AutoTokenizer

# 1. Text Engine (ClinicalBERT)
class TextEngine(nn.Module):
    def __init__(self, model_name="emilyalsentzer/Bio_ClinicalBERT"):
        super(TextEngine, self).__init__()
        try:
            self.bert = AutoModel.from_pretrained(model_name)
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        except:
            print("Warning: Could not load ClinicalBERT. Using DistilBERT.")
            self.bert = AutoModel.from_pretrained("distilbert-base-uncased")
            self.tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
            
        self.fc = nn.Linear(768, 512) 
        
    def get_tokenizer(self):
        return self.tokenizer

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :] 
        return self.fc(cls_token)

# 2. Time-Series Engine
class TimeSeriesEngine(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2):
        super(TimeSeriesEngine, self).__init__()
        self.lstm = nn.LSTM(input_dim, hidden_dim, num_layers, batch_first=True, bidirectional=True)
        # Note: Set batch_first=True for Transformer if possible or handle permutation
        self.transformer_layer = nn.TransformerEncoderLayer(d_model=hidden_dim*2, nhead=4, batch_first=True)
        self.transformer = nn.TransformerEncoder(self.transformer_layer, num_layers=1)
        self.fc = nn.Linear(hidden_dim*2, 512)

    def forward(self, x):
        # x: [Batch, 1, SeqLen] -> [Batch, SeqLen, 1]
        x = x.permute(0, 2, 1)
        r_out, _ = self.lstm(x) # [Batch, Seq, 128]
        
        # Transformer with batch_first=True
        t_out = self.transformer(r_out)
        
        # Pooling (Mean)
        pool = torch.mean(t_out, dim=1) # [Batch, 128]
        return self.fc(pool) # [Batch, 512]

# 3. Visual Engine (ResNet)
class VisualEngine(nn.Module):
    def __init__(self):
        super(VisualEngine, self).__init__()
        # Use updated weights parameter to avoid warnings
        try:
            weights = models.ResNet18_Weights.IMAGENET1K_V1
            resnet = models.resnet18(weights=weights)
        except:
            resnet = models.resnet18(pretrained=True) # Fallback for older torch
            
        self.features = nn.Sequential(*list(resnet.children())[:-1]) # Remove fc
        self.fc = nn.Linear(512, 512)
        
    def forward(self, x):
        f = self.features(x)
        f = f.view(f.size(0), -1)
        return self.fc(f)

# 4. Longitudinal Fusion Module
class LongitudinalFusion(nn.Module):
    def __init__(self, embed_dim=512):
        super(LongitudinalFusion, self).__init__()
        self.cross_modal_attn = nn.MultiheadAttention(embed_dim, num_heads=8, batch_first=True)
        self.norm = nn.LayerNorm(embed_dim)
        self.feed_forward = nn.Sequential(
            nn.Linear(embed_dim, embed_dim * 2),
            nn.ReLU(),
            nn.Linear(embed_dim * 2, embed_dim)
        )
        
    def forward(self, x):
        # x: [Batch, Seq(Modalities), Dim]
        attn_out, weights = self.cross_modal_attn(x, x, x, need_weights=True)
        x = self.norm(x + attn_out)
        return self.norm(x + self.feed_forward(x)), weights

# 5. Full HyCF-TFM Model
class HyCF_TFM(nn.Module):
    def __init__(self):
        super(HyCF_TFM, self).__init__()
        self.text_engine = TextEngine()
        self.time_engine = TimeSeriesEngine()
        self.visual_engine = VisualEngine()
        
        self.fusion = LongitudinalFusion(embed_dim=512)
        
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 1)
        )
        
    def forward(self, ecg, img, input_ids, mask):
        f_text = self.text_engine(input_ids, mask)
        f_time = self.time_engine(ecg)
        f_img  = self.visual_engine(img)
        
        stacked = torch.stack([f_text, f_time, f_img], dim=1) # [B, 3, 512]
        
        fused, weights = self.fusion(stacked)
        
        flat_fused = fused.view(fused.size(0), -1)
        logits = self.classifier(flat_fused)
        return logits, weights

## 3. 模型架构 (HyCF-TFM)

模型包含三个专有引擎和一个 Transformer 融合模块。

In [ ]:
class IntentAgent:
    def __init__(self, client):
        self.client = client
        self.system_prompt = """
        You are an expert Cardiologist AI Assistant. Analyze the multimodal data for heart failure prediction.
        Inputs: 
        1. Clinical Diagnosis (from Admission Records)
        2. CXR Risk Factors (Visual features implicitly analyzed)
        3. Clinical Notes (Discharge Summary Snippets)
        4. Predicted Risk Score (0-1)
        5. Modality Attention (Where the model focused: Text, ECG, or Image)
        
        Structure your response exactly as:
        **Assessment**: [Summary of risk and key clinical findings]
        **Key Evidence**: [Highlight specific terms from notes or signal anomalies]
        **Recommendation**: [Specific clinical action plan or follow-up]
        """

    def generate_report(self, sample_info, risk_score, attn_weights):
        # Clean up weights input
        try:
            if isinstance(attn_weights, torch.Tensor):
                weights = attn_weights.cpu().numpy().flatten()
            elif isinstance(attn_weights, np.ndarray):
                weights = attn_weights.flatten()
            elif isinstance(attn_weights, list):
                weights = np.array(attn_weights)
            else:
                weights = np.array([0.33, 0.33, 0.33])

            if len(weights) < 3:
                weights = np.pad(weights, (0, 3-len(weights)), 'constant')
            
            w_text, w_time, w_img = weights[0], weights[1], weights[2]
            
            focus = []
            if w_time > 0.35: focus.append(f"ECG Signal (Attn: {w_time:.2f})")
            if w_img > 0.35: focus.append(f"Chest X-Ray (Attn: {w_img:.2f})")
            if w_text > 0.35: focus.append(f"Clinical Notes (Attn: {w_text:.2f})")
            
            focus_str = ", ".join(focus) if focus else "Balanced multimodal evidence"
            
        except Exception as e:
            focus_str = "Multimodal analysis"

        # Safe extraction of text snippet
        text_snippet = str(sample_info.get('text', 'No notes provided'))[:500]

        user_prompt = f"""
        Patient Profile:
        - Primary Diagnosis: {sample_info.get('nyha', 'Unknown')}
        - Image Status: {sample_info.get('img_risk', 'Available')}
        - Key Clinical Notes: "{text_snippet}..."
        
        Model Output:
        - Readmission Risk Probability: {risk_score:.4f}
        - AI Attention Focus: {focus_str}
        """
        
        # print(f"Asking Agent (Model Focus: {focus_str})...")
        try:
            response = self.client.chat.completions.create(
                model="gpt-4o-mini",
                messages=[
                    {"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": user_prompt}
                ]
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"LLM Agent Error: {e}"

# Training Helpers
def train_one_epoch(model, loader, criterion, optimizer, device, tokenizer):
    model.train()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    for batch_idx, batch in enumerate(loader):
        ecg = batch['ecg'].to(device)
        img = batch['image'].to(device)
        targets = batch['label'].to(device).unsqueeze(1)
        
        texts = batch['text_raw'] # Should be tuple/list of strings
        inputs = tokenizer(list(texts), padding=True, truncation=True, return_tensors="pt", max_length=128)
        input_ids = inputs['input_ids'].to(device)
        mask = inputs['attention_mask'].to(device)
        
        optimizer.zero_grad()
        logits, _ = model(ecg, img, input_ids, mask)
        loss = criterion(logits, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        all_preds.extend(probs)
        all_targets.extend(targets.cpu().numpy())
        
    avg_loss = total_loss / len(loader)
    try:
        if len(np.unique(all_targets)) > 1:
            auc = roc_auc_score(all_targets, all_preds)
        else:
            auc = 0.5
    except:
        auc = 0.5
    return avg_loss, auc

def evaluate(model, loader, criterion, device, tokenizer):
    model.eval()
    total_loss = 0
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            ecg = batch['ecg'].to(device)
            img = batch['image'].to(device)
            targets = batch['label'].to(device).unsqueeze(1)
            
            texts = batch['text_raw']
            inputs = tokenizer(list(texts), padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = inputs['input_ids'].to(device)
            mask = inputs['attention_mask'].to(device)
            
            logits, weights = model(ecg, img, input_ids, mask)
            loss = criterion(logits, targets)
            total_loss += loss.item()
            
            probs = torch.sigmoid(logits).cpu().numpy()
            all_preds.extend(probs)
            all_targets.extend(targets.cpu().numpy())
            
    avg_loss = total_loss / len(loader) if len(loader) > 0 else 0
    try:
        if len(np.unique(all_targets)) > 1:
            auc = roc_auc_score(all_targets, all_preds)
            ap = average_precision_score(all_targets, all_preds)
            f1 = f1_score(all_targets, (np.array(all_preds)>0.5).astype(int))
        else:
            auc, ap, f1 = 0.5, 0.0, 0.0
    except:
        auc, ap, f1 = 0.5, 0.0, 0.0
        
    return avg_loss, auc, ap, f1

## 4. 训练与评估循环

定义训练步骤和评估指标计算。

In [ ]:
import warnings
warnings.filterwarnings("ignore") # Suppress PyTorch warnings for demo clarity

# Initialize Model
print("Initializing HyCF-TFM Model...")
model = HyCF_TFM().to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# Tokenizer
tokenizer = model.text_engine.get_tokenizer()

# Training Loop
EPOCHS = 3
print(f"Starting Training for {EPOCHS} epochs on {device}...")

for epoch in range(EPOCHS):
    train_loss, train_auc = train_one_epoch(model, train_loader, criterion, optimizer, device, tokenizer)
    val_loss, val_auc, val_ap, val_f1 = evaluate(model, test_loader, criterion, device, tokenizer)
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {train_loss:.4f} AUC: {train_auc:.4f} | Val Loss: {val_loss:.4f} AUC: {val_auc:.4f} F1: {val_f1:.4f}")

# Save Model
torch.save(model.state_dict(), "hycf_tfm_model.pth")
print("Model saved.")

# --- Case Study with Intent Agent ---
print("\n--- Running Clinical Case Study with Intent Agent ---")
try:
    agent = IntentAgent(client)

    # Pick a High Risk Sample from Test Set
    model.eval()
    found_case = False

    with torch.no_grad():
        for batch in test_loader:
            if found_case: break
            targets = batch['label']
            for i in range(len(targets)):
                if targets[i] == 1.0: # High Risk
                    # Extract Single Sample Data
                    ecg = batch['ecg'][i].unsqueeze(0).to(device)
                    img = batch['image'][i].unsqueeze(0).to(device)
                    txt = batch['text_raw'][i]
                    
                    # Model Inference
                    inputs = tokenizer([txt], padding=True, return_tensors="pt")
                    input_ids = inputs['input_ids'].to(device)
                    mask = inputs['attention_mask'].to(device)
                    
                    logits, weights = model(ecg, img, input_ids, mask)
                    prob = torch.sigmoid(logits).item()
                    
                    # Process Weights: [Batch, Heads, Target, Source]
                    # weights are [1, 8, 3, 3] -> Average heads first
                    avg_weights_matrix = weights.mean(dim=1).squeeze(0) # [3, 3]
                    
                    # IMPORTANT: Matrix is [Target, Source] usually. 
                    # We want to know how much Text/Time/Img contributed to the Final Fusion Representation.
                    # Summing columns? Or just diagonal?
                    # Let's take global attention received by each mod.
                    modality_importance = avg_weights_matrix.mean(dim=0).cpu().numpy() # [3]
                    
                    # Normalize
                    total = modality_importance.sum() + 1e-9
                    modality_importance = modality_importance / total
                    
                    # Sample Info
                    nyha_val = batch['nyha'][i].item()
                    img_risk = batch['img_risk'][i].item()
                    
                    sample_info = {
                        "nyha": f"Class {nyha_val}",
                        "img_risk": "Yes" if img_risk else "No",
                        "text": txt
                    }
                    
                    # Generate Report
                    print(f"Generating Report for Case (Prob: {prob:.4f})...")
                    # Pass the array safely
                    report = agent.generate_report(sample_info, prob, modality_importance)
                    print("\n" + "="*40)
                    print(report)
                    print("="*40 + "\n")
                    
                    # Save for Visualizations later
                    avg_weights = avg_weights_matrix.cpu().numpy()
                    
                    found_case = True
                    break

    if not found_case:
        print("No high risk cases found in this batch for Case Study.")
except Exception as e:
    print(f"Case Study Failed: {e}")
    # Create dummy weights for visualization block if failed
    avg_weights = np.eye(3)

In [ ]:
# --- Real-World Experimental Visualization ---
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns

def evaluate_variant(model, loader, device, tokenizer, variant="full"):
    """
    Evaluates the model on the test set with specific modalities masked to simulate ablation.
    variant: "full", "no_text", "no_ecg", "no_image"
    """
    model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in loader:
            ecg = batch['ecg'].to(device)
            img = batch['image'].to(device)
            targets = batch['label'].to(device).unsqueeze(1)
            
            texts = batch['text_raw']
            inputs = tokenizer(texts, padding=True, truncation=True, return_tensors="pt", max_length=128)
            input_ids = inputs['input_ids'].to(device)
            mask = inputs['attention_mask'].to(device)
            
            # --- APPLY ABLATION MASKS ---
            if variant == "no_text":
                # Mask text by passing zeros (pad tokens) output from empty bert or zero embeddings
                # Simplest way: Zero out the attention mask so nothing is attended to
                mask = torch.zeros_like(mask)
            
            if variant == "no_ecg":
                ecg = torch.zeros_like(ecg)
            
            if variant == "no_image":
                img = torch.zeros_like(img)
            # ---------------------------

            logits, _ = model(ecg, img, input_ids, mask)
            probs = torch.sigmoid(logits).cpu().numpy()
            
            all_preds.extend(probs)
            all_targets.extend(targets.cpu().numpy())
            
    try:
        if len(np.unique(all_targets)) > 1:
            score = roc_auc_score(all_targets, all_preds)
        else:
            score = 0.5
    except:
        score = 0.5
        
    return score, all_targets, all_preds

print("Running Real-Data Ablation Study (Inference Time)...")
variants = ["Full HyCF-TFM", "w/o Text", "w/o ECG", "w/o Image"]
scores = []
predictions = {} # Store for ROC Plot

# 1. Evaluate Variants
score_full, y_true, y_pred_full = evaluate_variant(model, test_loader, device, tokenizer, "full")
scores.append(score_full)
predictions["Full HyCF-TFM"] = (y_true, y_pred_full)

score_no_text, _, _ = evaluate_variant(model, test_loader, device, tokenizer, "no_text")
scores.append(score_no_text)

score_no_ecg, _, _ = evaluate_variant(model, test_loader, device, tokenizer, "no_ecg")
scores.append(score_no_ecg)

score_no_img, _, _ = evaluate_variant(model, test_loader, device, tokenizer, "no_image")
scores.append(score_no_img)

# --- Plotting ---
plt.figure(figsize=(18, 5))

# Plot 1: Ablation Bar Chart
plt.subplot(1, 3, 1)
sns.barplot(x=variants, y=scores, palette="viridis")
plt.title(f"Ablation Study (Actual Data)\nImpact of Modality Removal on AUC", fontsize=12)
plt.ylabel("ROC AUC Score")
plt.ylim(0.0, 1.1)
for i, v in enumerate(scores):
    plt.text(i, v + 0.02, f"{v:.3f}", ha='center', fontweight='bold')

# Plot 2: ROC Curve (Full Model)
plt.subplot(1, 3, 2)
fpr, tpr, _ = roc_curve(y_true, y_pred_full)
roc_auc = auc(fpr, tpr)
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([-0.05, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (Test Set)')
plt.legend(loc="lower right")

# Plot 3: Confusion Matrix
plt.subplot(1, 3, 3)
y_pred_binary = (np.array(y_pred_full) > 0.5).astype(int)
cm = confusion_matrix(y_true, y_pred_binary)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.xticks([0.5, 1.5], ["Low Risk", "High Risk"])
plt.yticks([0.5, 1.5], ["Low Risk", "High Risk"])

plt.tight_layout()
plt.show()

# --- Separate Plot: Patient Trajectory with Attention ---
def plot_real_trajectory_with_context():
    # Find a high risk case in test set
    found = False
    for batch in test_loader:
        if found: break
        for i in range(len(batch['label'])):
            if batch['label'][i] == 1:
                ecg_np = batch['ecg'][i].flatten().numpy()
                txt = batch['text_raw'][i][:60] + "..."
                
                plt.figure(figsize=(12, 4))
                plt.plot(ecg_np[:1000], color='black', lw=0.8) # First 1000 pts
                
                # Highlight regions (Simulated Attention for demo visualization)
                # In real scenario, would project Attention weights back to time steps
                plt.axvspan(200, 400, color='red', alpha=0.1, label="High Risk Region (Simulated)")
                plt.axvspan(700, 800, color='red', alpha=0.1)
                
                plt.title(f"Patient ECG Trajectory (Actual Sample)\nClinical Context: {txt}")
                plt.xlabel("Time Steps")
                plt.ylabel("Normalized Amplitude")
                plt.legend(loc="upper right")
                plt.tight_layout()
                plt.show()
                found = True
                break
plot_real_trajectory_with_context()